In [ ]:
# THIS WAS RUN IN COLAB; training interrupted after about 2 hours

In [1]:
! pip install mace-torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.1/453.1 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 71.1 MB/s eta 0:00:00
  Created wheel for python-hostlist: filename=python_hostlist-2.3.0-py3-none-any.whl size=39449 sha256=50ee7fd03cd61fcb9a934bf1cda9abe13a46ff6020c030180a287e62d5975de7
  Stored in directory: /root/.cache/pip/wheels/02/e4/34/75fc0cd5b7889d8cc4ce6fb2f74c9fd17b3c6138cb03832481
Successfully built python-hostlist


In [2]:
import sys
import os
from mace.cli.run_train import main

/usr/local/lib/python3.12/dist-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [19]:
# import importlib
# import useful_functions
# importlib.reload(useful_functions)
from useful_functions import clean_dataset, split_train_valid # custom functions for easier re-runs

In [9]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

In [10]:
from google.colab import files

uploaded = files.upload()

Saving OUTCAR to OUTCAR


In [11]:
my_full_file="./new_data/1.NVT_300/A.1_10K/cleaned_dataset.extxyz" # not yet created
my_train_file="./new_data/1.NVT_300/A.1_10K/train.extxyz" # not yet created
my_valid_file="./new_data/1.NVT_300/A.1_10K/valid.extxyz" # not yet created

my_sim_checkpoints_dir="./new_simulation/1.NVT_300/A.1_10K/checkpoints"
my_sim_results_dir="./new_simulation/1.NVT_300/A.1_10K/results"

outcar_file= "./OUTCAR"  # ./new_data/1.NVT_300/A.1_10K/OUTCAR" # unzipped (one line from terminal)

In [13]:
os.makedirs(f"{my_sim_checkpoints_dir}", exist_ok=True)
os.makedirs(f"{my_sim_results_dir}", exist_ok=True)


In [15]:
!mkdir new_data

In [16]:
!mkdir new_data/1.NVT_300

In [17]:
!mkdir new_data/1.NVT_300/A.1_10K

In [20]:
clean_dataset(
    outcar_file=outcar_file,
    destination_file=my_full_file,
    start_idx=6840 # from convergence check (visually preferred to 8400)
)

Retrieved 3160 structures from OUTCAR.
File saved as ./new_data/1.NVT_300/A.1_10K/cleaned_dataset.extxyz


0

In [21]:
split_train_valid(
    full_cleaned_extxyz_file=my_full_file,
    destination_train_file=my_train_file,
    destination_valid_file=my_valid_file,
    train_frac=0.9,
)

Total frames uploaded: 3160
Saved 2844 frames in ./new_data/1.NVT_300/A.1_10K/train.extxyz and 316 in ./new_data/1.NVT_300/A.1_10K/valid.extxyz


0

In [28]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [29]:
# hyperpars for MACE training (sources: MACE T01 tutorial; Claude AI targeted help)
custom_args = [
    "mace_run_training",
    "--name=model_3000_pts",
    f"--train_file={my_train_file}",
    f"--valid_file={my_valid_file}",
    "--energy_key=energy",
    "--forces_key=forces",
    "--E0s=foundation", # try "isolated" after adding isolated atom energies to training set?
               #"mp" means from materials project (used to train foundation model)
    #"--model=MACE",
    #"--num_interactions=2", # keep default
    "--max_num_epochs=50",
    "--patience=10",
    "--batch_size=5",       # was 10
    "--valid_batch_size=5", # was 10
    "--device=cuda",
    "--default_dtype=float32", # less memory than 64
    "--r_max=5.0",
    f"--checkpoints_dir={my_sim_checkpoints_dir}",
    f"--results_dir={my_sim_results_dir}",
    "--keep_checkpoints",

    "--num_channels=64",
    "--max_L=0",
    "--seed=1",

    # use a pre-trained ("foundation") model:
    "--foundation_model=small",           # Use MACE‑MP‑0 small
    "--multiheads_finetuning=True",        # Enable recommended Multihead Replay (for generalizable models; but expensive)
    "--pt_train_file=mp",                 # Required for Multihead Replay

]

sys.argv = custom_args

In [30]:
if __name__ == "__main__":
    try:
        main()
        print("Training completed!")
    except Exception as e:
        print(f"Error: {e}")

INFO:root:===========VERIFYING SETTINGS===========


2026-08-08 20:26:52.166 INFO: ===========VERIFYING SETTINGS===========
2026-08-08 20:26:52.166 INFO: ===========VERIFYING SETTINGS===========
2026-08-08 20:26:52.166 INFO: ===========VERIFYING SETTINGS===========
2026-08-08 20:26:52.166 INFO: ===========VERIFYING SETTINGS===========


INFO:root:MACE version: 0.3.16


2026-08-08 20:26:52.172 INFO: MACE version: 0.3.16
2026-08-08 20:26:52.172 INFO: MACE version: 0.3.16
2026-08-08 20:26:52.172 INFO: MACE version: 0.3.16
2026-08-08 20:26:52.172 INFO: MACE version: 0.3.16


DEBUG:root:Configuration: Namespace(config=None, name='model_3000_pts', seed=1, work_dir='.', log_dir='./logs', model_dir='.', checkpoints_dir='./new_simulation/1.NVT_300/A.1_10K/checkpoints', results_dir='./new_simulation/1.NVT_300/A.1_10K/results', downloads_dir='./downloads', device='cuda', default_dtype='float32', distributed=False, launcher='slurm', log_level='INFO', plot=True, plot_frequency=0, plot_interaction_e=False, error_table='PerAtomRMSE', model='MACE', r_max=5.0, radial_type='bessel', num_radial_basis=8, num_cutoff_basis=5, pair_repulsion=False, distance_transform='None', apply_cutoff=True, use_last_readout_only=False, use_embedding_readout=False, interaction='RealAgnosticResidualInteractionBlock', interaction_first='RealAgnosticResidualInteractionBlock', max_ell=3, correlation=3, use_reduced_cg=False, use_so3=False, use_agnostic_product=False, num_interactions=2, MLP_irreps='16x0e', radial_MLP='[64, 64, 64]', hidden_irreps=64x0e, edge_irreps=None, use_edge_irreps_first=F

...

2026-08-08 22:24:38.123 INFO: Epoch 17: head: pt_head, loss=0.01074218, RMSE_E_per_atom=  117.14 meV, RMSE_F=  195.87 meV / A, RMSE_stress=    6.66 meV / A^3
2026-08-08 22:24:38.123 INFO: Epoch 17: head: pt_head, loss=0.01074218, RMSE_E_per_atom=  117.14 meV, RMSE_F=  195.87 meV / A, RMSE_stress=    6.66 meV / A^3
2026-08-08 22:24:38.123 INFO: Epoch 17: head: pt_head, loss=0.01074218, RMSE_E_per_atom=  117.14 meV, RMSE_F=  195.87 meV / A, RMSE_stress=    6.66 meV / A^3
2026-08-08 22:24:38.123 INFO: Epoch 17: head: pt_head, loss=0.01074218, RMSE_E_per_atom=  117.14 meV, RMSE_F=  195.87 meV / A, RMSE_stress=    6.66 meV / A^3


INFO:root:Epoch 17: head: Default, loss=0.00534080, RMSE_E_per_atom=    0.93 meV, RMSE_F=   44.31 meV / A, RMSE_stress=None meV / A^3


2026-08-08 22:24:44.764 INFO: Epoch 17: head: Default, loss=0.00534080, RMSE_E_per_atom=    0.93 meV, RMSE_F=   44.31 meV / A, RMSE_stress=None meV / A^3
2026-08-08 22:24:44.764 INFO: Epoch 17: head: Default, loss=0.00534080, RMSE_E_per_atom=    0.93 meV, RMSE_F=   44.31 meV / A, RMSE_stress=None meV / A^3
2026-08-08 22:24:44.764 INFO: Epoch 17: head: Default, loss=0.00534080, RMSE_E_per_atom=    0.93 meV, RMSE_F=   44.31 meV / A, RMSE_stress=None meV / A^3
2026-08-08 22:24:44.764 INFO: Epoch 17: head: Default, loss=0.00534080, RMSE_E_per_atom=    0.93 meV, RMSE_F=   44.31 meV / A, RMSE_stress=None meV / A^3


DEBUG:root:Saving checkpoint: ./new_simulation/1.NVT_300/A.1_10K/checkpoints/model_3000_pts_run-1_epoch-17.pt


KeyboardInterrupt: 

In [33]:
files.download("/content/new_simulation/1.NVT_300/A.1_10K/checkpoints/model_3000_pts_run-1_epoch-17.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
files.download("/content/model_3000_pts_valid_indices_1.txt")
files.download("/content/mp_finetuning-model_3000_pts_run-1_combined.xyz")

# others can be downloaded manually

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# RMSE_F (forces): Default head - for MD-quality models, sub-50 meV/Å is commonly considered good, sub-20-30 meV/Å is very good.
# RMSE_E_per_atom (energy): Default head's sub-few-meV/atom is typical for usable models.

In [ ]:
# TO DO: a) fix warnings (expecially "energy" and "forces" keys),
# b) test??